# CivicPulse AI: RDD2022 Road Defect Model Training & ONNX Export
**Course / Portfolio Context:** B.E. AI & Data Science Capstone Project
**Target Architecture:** YOLOv8n (Ultralytics) $\rightarrow$ ONNX Runtime (CPU/Edge Deployment)

This notebook trains a localized road defect detector on the Indian road condition subset of the **Road Damage Dataset 2022 (RDD2022)**.

### Dataset Attribution & Licence
- **Dataset Source:** Road Damage Detection Challenge 2022 (RDD2022), Sekilab Global ([sekilab/RoadDamageDetector](https://github.com/sekilab/RoadDamageDetector)).
- **Licence:** Creative Commons Attribution-ShareAlike 4.0 International (CC BY-SA 4.0).
- **Classes Detected:**
  1. `D00`: Longitudinal Crack (id: 0)
  2. `D10`: Transverse Crack (id: 1)
  3. `D20`: Alligator Crack (id: 2)
  4. `D40`: Pothole (id: 3)

### Why Google Colab?
Local development is performed on an Intel CPU without a dedicated CUDA GPU. This notebook runs on a free Google Colab T4 GPU instance, executes reproducible training with a fixed seed (`seed=42`), exports the trained model to standard ONNX (`rdd2022_yolov8n.onnx`), and downloads it for deployment in the CivicPulse FastAPI backend at `models/rdd2022_yolov8n.onnx`.

## 1. Hardware & GPU Verification

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

## 2. Environment Setup & Dependency Installation

In [ ]:
!pip install -q ultralytics onnx onnxruntime open-clip-torch pyyaml tqdm scikit-learn matplotlib

## 3. Clone Repository or Set Up Workspace

In [ ]:
# If running directly inside a cloned repository:
# !git clone https://github.com/yourusername/CivicPulse-AI.git
# %cd CivicPulse-AI

import os
print(f"Current working directory: {os.getcwd()}")

## 4. Download and Prepare RDD2022 India Dataset
We use grouped sequence splitting (`ml/dataset.py`) to prevent consecutive dashcam frames from leaking between train (70%), val (15%), and test (15%) splits.

In [ ]:
# Create data directory
!mkdir -p data/raw/rdd2022

# Download official RDD2022 India subset (~1.2 GB compressed)
# Licence: CC BY-SA 4.0 (Sekilab Global)
!wget -q -O rdd2022_india.tar.gz "https://bigdata.sekilab.global/cdd2022/RDD2022_India.tar.gz"
!tar -xzf rdd2022_india.tar.gz -C data/raw/rdd2022/

# Run anti-leakage dataset preparation script
!python ml/dataset.py --raw-dir data/raw/rdd2022 --output-dir data/rdd2022_india

## 5. Train YOLOv8n with Fixed Seed & Deterministic Flags

In [ ]:
# Train for 30 epochs with fixed seed=42
!python ml/train.py \
    --config ml/configs/rdd2022_yolov8n.yaml \
    --epochs 30 \
    --batch-size 16 \
    --imgsz 640 \
    --seed 42 \
    --output-dir runs/train \
    --export-onnx

## 6. Verify Numeric Parity (PyTorch vs ONNX Runtime)

In [ ]:
from ml.train import verify_onnx_parity
from ultralytics import YOLO

best_pt = "runs/train/rdd2022_experiment/weights/best.pt"
best_onnx = "runs/train/rdd2022_experiment/weights/best.onnx"

pt_model = YOLO(best_pt).model
parity = verify_onnx_parity(pt_model, best_onnx)
print("ONNX Parity Verification:", parity)

## 7. Evaluate Detector on Test Split & Out-of-Distribution Bengaluru Test Set

In [ ]:
!python ml/eval/evaluate_detector.py --model runs/train/rdd2022_experiment/weights/best.onnx --test-dir data/rdd2022_india/images/test
!python ml/eval/evaluate_ood.py

## 8. Download Trained ONNX Model for Local Deployment

In [ ]:
from google.colab import files
import os
import shutil

onnx_export_path = "runs/train/rdd2022_experiment/weights/best.onnx"
if os.path.exists(onnx_export_path):
    output_filename = "rdd2022_yolov8n.onnx"
    shutil.copy2(onnx_export_path, output_filename)
    print(f"Downloading {output_filename}...")
    files.download(output_filename)
    print("\nDone! Move this downloaded file to 'models/rdd2022_yolov8n.onnx' in your local project root.")
else:
    print("Trained ONNX file not found. Please verify training completed successfully.")